# Tabular Deep Learning with Embeddings

This notebook is the recruiter-facing analytical walkthrough. Production code lives in `src/`, the hosted app lives in `app/`, and saved artifacts live in `models/`.

## Objective

Build a binary classifier that learns dense vectors for categorical variables and combines them with scaled numerical features. The dataset is deterministic and synthetic so the full project can be reproduced without exposing private data.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import *
from src.data_generation import generate_synthetic_tabular_data
from src.data_preprocessing import split_dataset, fit_all_preprocessors, build_model_inputs
from src.model_training import build_tabular_embedding_model, compile_binary_model, compute_class_weights, train_embedding_model
from src.model_evaluation import evaluate_binary_classifier, select_f1_threshold

## 1. Generate and inspect data

In [ ]:
df = generate_synthetic_tabular_data(n_samples=25_000, seed=SEED)
print(df.shape)
df.head()

In [ ]:
df[TARGET_COLUMN].value_counts(normalize=True).rename("share").to_frame()

## 2. Leakage-aware split and preprocessing

Numerical preprocessors and categorical vocabularies are fitted on training data only. The safe categorical encoder reserves index 0 for unseen values.

In [ ]:
splits = split_dataset(df, TARGET_COLUMN, seed=SEED)
encoders, imputer, scaler = fit_all_preprocessors(
    splits.train, CATEGORICAL_FEATURES, NUMERICAL_FEATURES
)

train_inputs, _ = build_model_inputs(splits.train, CATEGORICAL_FEATURES, NUMERICAL_FEATURES, encoders, imputer, scaler)
val_inputs, _ = build_model_inputs(splits.validation, CATEGORICAL_FEATURES, NUMERICAL_FEATURES, encoders, imputer, scaler)
test_inputs, _ = build_model_inputs(splits.test, CATEGORICAL_FEATURES, NUMERICAL_FEATURES, encoders, imputer, scaler)

[len(splits.train), len(splits.validation), len(splits.test)]

## 3. Build the embedding ANN

In [ ]:
model = build_tabular_embedding_model(CATEGORICAL_FEATURES, encoders, len(NUMERICAL_FEATURES))
compile_binary_model(model)
model.summary()

## 4. Train

Early stopping monitors validation AUC and class weights reduce majority-class dominance.

In [ ]:
history = train_embedding_model(
    model,
    train_inputs, splits.train[TARGET_COLUMN].to_numpy(dtype="float32"),
    val_inputs, splits.validation[TARGET_COLUMN].to_numpy(dtype="float32"),
    compute_class_weights(splits.train[TARGET_COLUMN]),
)

## 5. Select threshold on validation data

In [ ]:
val_prob = model.predict(val_inputs, verbose=0).reshape(-1)
best_threshold, threshold_table = select_f1_threshold(
    splits.validation[TARGET_COLUMN].to_numpy(), val_prob
)
best_threshold, threshold_table.sort_values("f1", ascending=False).head()

## 6. Final held-out test evaluation

In [ ]:
test_prob = model.predict(test_inputs, verbose=0).reshape(-1)
metrics = evaluate_binary_classifier(
    splits.test[TARGET_COLUMN].to_numpy(), test_prob, threshold=0.50
)
pd.DataFrame([{k: v for k, v in metrics.items() if isinstance(v, (int, float))}])

## 7. Save production artifacts

Use `python train_model.py` for the complete saving, baseline comparison, plot generation, and metadata workflow.

In [ ]:
# Production entrypoint
# !python train_model.py